In [0]:
import dlt
from pyspark.sql.functions import *
import spark

@dlt.table(
  name="sales_gold",
  comment="Aggregated sales"
)
def sales_gold():
    df = dlt.read("sales_silver")

    return (
        df.groupBy("product")
          .agg(sum("amount").alias("total_sales"))
    )

In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="analytics.gold.sales_gold",
    comment="Aggregated sales metrics"
)
def sales_gold():
    df = spark.read.table("dev.silver.sales_silver")

    return (
        df.groupBy("product")
          .sum("amount")
    )

### Gold Layer (Business Metrics + Strict Rules)
Gold layer typically has business validation rules.

In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="analytics.gold.sales_gold",
    comment="Business sales metrics"
)
@dlt.expect_or_fail("valid_sales_value", "total_sales >= 0")
def sales_gold():

    df = spark.read.table("dev.silver.sales_silver")

    return (
        df.groupBy("product")
          .agg(sum("amount").alias("total_sales"))
    )

### Gold Layer (Business Aggregations)
Gold provides analytics-ready tables.

In [0]:
@dlt.table(
    name="analytics.gold.sales_gold",
    comment="Aggregated sales metrics"
)
def sales_gold():
    df = dlt.read("analytics.silver.sales_silver")

    return (
        df.filter(col("__CURRENT") == True)
        .groupBy(
            col("product"),
            to_date(col("updated_at")).alias("sale_date")
        )
        .agg(
            sum("amount").alias("total_sales"),
            count("order_id").alias("total_orders")
        )
    )